# ema-first-moment composite — cx20: m EMA then bias-correct: m_hat = m / (1 - beta1**t)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-first-moment`, `bias-correction-divide`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-first-moment"
DD_ATOM_IDS = ["ema-first-moment", "bias-correction-divide"]
DD_SUBTOPICS = ["Optimizer: Adam EMA first moment", "Optimizer: Adam bias-correction divide"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Adam's first-moment EMA starts from zero. The recurrence `m = beta1 * m + (1 - beta1) * g` therefore underestimates the true mean of `g` for the first few steps — for constant `g`, after `t` steps you get `m_t = (1 - beta1**t) * g`.

The fix is the **bias-correction divide**:
```
m_hat = m / (1 - beta1**t)
```
With constant `g`, this recovers `m_hat = g` EXACTLY at every step — the divide undoes the warmup shrinkage. As `t -> inf`, `beta1**t -> 0`, so the divisor `-> 1` and the correction fades to a no-op.

**Composition.** This drill chains the two atoms: take a gradient `g`, drive one step of the `m` EMA from the current buffer, THEN apply the bias-correction divide. The combined function returns BOTH the updated `m` (caller stores it for the next step) and `m_hat` (used downstream in the parameter update).

**Why `t` is 1-based.** At `t=0`, `beta1**0 = 1` and the divisor is 0 — division by zero. Adam's step counter starts at 1 and is incremented BEFORE the bias correction divide. Pass `t=0` and you get NaN.

**Anatomy.**
```python
def step(m, g, beta1, t):
    m_new = beta1 * m + (1 - beta1) * g    # atom A
    m_hat = m_new / (1 - beta1 ** t)        # atom B
    return m_new, m_hat
```

### Composite Exercise — m EMA then bias-correct: m_hat = m / (1 - beta1**t)

**Atoms exercised together**: `ema-first-moment`, `bias-correction-divide`

Implement `cx20_ema_then_bias_correct(m, g, beta1, t_step)`.

Inputs:
- `m`: current first-moment buffer (Tensor of any shape).
- `g`: current gradient (Tensor, same shape as `m`).
- `beta1`: float decay (e.g. 0.9).
- `t_step`: int >= 1, the 1-based step counter.

Returns a tuple `(m_new, m_hat)`:
- `m_new = beta1 * m + (1 - beta1) * g` — the updated buffer (atom: ema-first-moment).
- `m_hat = m_new / (1 - beta1 ** t_step)` — bias-corrected (atom: bias-correction-divide).

Do NOT mutate the input `m` in place — return fresh tensors. (The caller is responsible for swapping `m` with `m_new` between steps.)

Sanity properties the test verifies:
- Step 1 with `m = 0`: `m_new = (1 - beta1) * g`, and `m_hat = g` EXACTLY (closed form).
- Many steps of constant `g` starting from zero: `m_hat` stays equal to `g` at every step.
- As `t_step -> inf`, the divisor -> 1, so `m_hat -> m_new`.
- The input `m` is not mutated.

In [ ]:
def cx20_ema_then_bias_correct(m, g, beta1, t_step):
    # Atom A (ema-first-moment): one EMA step on m.
    m_new = beta1 * m + (1.0 - beta1) * g
    # Atom B (bias-correction-divide): undo the zero-init bias by dividing by (1 - beta1^t).
    m_hat = m_new / (1.0 - beta1 ** t_step)
    return m_new, m_hat


<details><summary>Show solution — cx20</summary>

```python
def cx20_ema_then_bias_correct(m, g, beta1, t_step):
    # Atom A (ema-first-moment): one EMA step on m.
    m_new = beta1 * m + (1.0 - beta1) * g
    # Atom B (bias-correction-divide): undo the zero-init bias by dividing by (1 - beta1^t).
    m_hat = m_new / (1.0 - beta1 ** t_step)
    return m_new, m_hat
```

**Two outputs, not one.** The caller needs `m_new` to seed the next step's EMA AND `m_hat` for the current parameter update. Returning only `m_hat` would lose the uncorrected buffer state. Returning only `m_new` would skip bias correction — Adam's first few steps would crawl.

**Don't mutate `m`.** Some impls write `m.mul_(beta1).add_(g, alpha=1-beta1)` to save an allocation. That's a valid optimization, but the caller-visible contract is identical: the returned tuple's first element IS the new buffer.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx20'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx20',
        'subtopics': ["Optimizer: Adam EMA first moment", "Optimizer: Adam bias-correction divide"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()